# Transaction Classification on the Elliptic++ Actor Graph (causal)

End-to-end notebook for the **transactions** classification task, exploiting the heterogeneous (wallet, tx) actor graph as the bridge that carries cross-timestep information. Every model is trained and evaluated under strict temporal causality: for a transaction observed at time $t$, only data with timestep $\le t$ enters the model.

## Why go through the actor graph
The transactions-only graph in `txs_edgelist.csv` contains 234,355 tx↔tx edges, **every one of which connects two transactions in the same timestep** (verified directly on the data). So no information can flow across timesteps in the transactions graph alone. The bipartite addr↔tx edges from the actor sub-dataset (477K input + 837K output) are the **only** edges that cross timesteps in this dataset: a wallet active at $t = 5,\,12,\,20$ is the bridge through which transaction $T$ at $t=20$ can see anything that happened at $t=5$ or $t=12$.

We exploit that bridge by lifting transaction classification into queries over the **causal trajectories** of the target transaction's incident wallets.

## Models
1. **Random Forest** baseline on the 165 per-tx features.
2. **Logistic Regression** baseline on the same.
3. **Temporal Transformer over wallet trajectories**, on top of a **causally-trained heterogeneous GNN embedder** (SAGEConv on tx ↔ tx and tx ↔ wallet relations, per-snapshot edge masks, supervised on tx labels at exactly `t` for `t ≤ TRAIN_END`).

## Data
- `transactions_data/txs_features.csv`: 203,769 txs × 165 features (28 local + 137 aggregate), each anchored to one timestep ∈ [1, 49].
- `transactions_data/txs_classes.csv`: per-tx labels — 4,545 illicit (`class=1`), 42,019 licit (`class=2`), 157,205 unknown (`class=3`).
- `transactions_data/txs_edgelist.csv`: 234K tx↔tx edges, **all within-timestep**.
- `actors_data/{AddrTx,TxAddr}_edgelist.txt`: bipartite addr↔tx edges (these inherit the tx endpoint's timestep).

## Temporal split (paper protocol with a held-out validation slice)
- **Train**: labelled txs with `t ≤ 29`
- **Val**: labelled txs with `30 ≤ t ≤ 34` (used for early stopping / threshold calibration)
- **Test**: labelled txs with `t ≥ 35`

The standard Elliptic split is t ≤ 34 train / t ≥ 35 test. We carve a 5-timestep validation slice from the end of train so we can pick best-epoch checkpoints honestly. The notable t = 43 distribution-shift cliff therefore lives entirely in test.

## What we **do not** use, and why
- **`wallets_features.txt`** is excluded entirely. Each wallet's 55-feature vector is byte-for-byte identical across all of its timesteps (verified directly), which means every entry is a **lifetime aggregate that incorporates post-t observations**. Any use of it inside a causal classifier leaks future information.
- **`AddrAddr_edgelist.txt`** is excluded. These wallet↔wallet edges have **no timestamp**, so we can't tell whether any specific A↔B link reflects a tx at t' ≤ t or t' > t. The bipartite addr↔tx edges already carry all the time-localised wallet–wallet connectivity we need (via 2-hop walks tx → wallet → other tx).

## Causality invariants enforced everywhere
1. The hetero-embedder forward pass at snapshot `t` uses an **edge mask** that drops every edge with a tx endpoint at `t' > t`, and **zeros tx input features** for txs with `t' > t`. So no information from future txs can reach any node in the embedded graph.
2. The embedder is trained by iterating `t = 1..TRAIN_END` and supervising on tx labels for txs at exactly `t`. The loss never observes labels with `t' > t`.
3. The wallet-trajectory cache stores embeddings only at `(wallet, t')` pairs where the wallet was incident to some tx at exactly `t'`.
4. The temporal transformer's per-tx token sequence for a target tx at `t` includes only wallet-trajectory tokens with `t' ≤ t`.
5. The `StandardScaler` is fitted on **train timesteps only**.

## Configurable hyperparameters
All hyperparameters live in the **config block** at the top of the imports cell. To resize the embedder, edit `EMBEDDER_*`. To change the trajectory window or the transformer, edit `MAX_*` and `TEMPORAL_*`.


In [ ]:
!pip install -q \
  "numpy<2" \
  "torch-geometric>=2.4.0" \
  "scikit-learn>=1.3.0" \
  "pandas>=2.0.0" \
  "matplotlib>=3.7.0" \
  "seaborn>=0.12.0" \
  "pyyaml>=6.0" \
  "tqdm>=4.65.0"


In [ ]:
!pip install -q --force-reinstall \
  "numpy<2" \
  "pandas>=2.0,<2.2.3" \
  "scikit-learn>=1.3.0" \
  "torch-geometric>=2.4.0"

import os
os.kill(os.getpid(), 9)


In [ ]:
from pathlib import Path
import warnings, time, bisect
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, roc_auc_score, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, GCNConv
from torch_geometric.transforms import ToUndirected

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = Path("/content/drive/MyDrive/stat175-final-project")
TRANSACTIONS_DIR = ROOT / "data"
WALLETS_DIR      = ROOT / "actor_data"

# ============================================================
#  CONFIG  — change here, then re-run
# ============================================================
N_TIME_STEPS    = 49
TRAIN_END       = 29        # train txs:  t in [1, 29]
VAL_END         = 34        # val txs:    t in [30, 34]
                            # test txs:   t in [35, 49]
RANDOM_SEED     = 42

# Hetero GNN embedder (causal). Conv type is switchable like in wallet_sota.ipynb:
#   "sage" -> SAGEConv on every relation (works for same-type and bipartite)
#   "gcn"  -> GCNConv on same-type relations (here: tx<->tx) and SAGEConv on bipartite
#            (vanilla GCN is undefined on bipartite)
EMBEDDER_CONV_TYPE    = "sage"     # "sage" or "gcn"
EMBEDDER_HIDDEN_DIM   = 64         # reduced from 128 (overfitting fix)
EMBEDDER_NUM_LAYERS   = 2
EMBEDDER_DROPOUT      = 0.5        # raised from 0.3 (overfitting fix)
EMBEDDER_LR           = 1e-3
EMBEDDER_WEIGHT_DECAY = 5e-4
EMBEDDER_EPOCHS       = 10         # reduced from 15 (train loss collapses fast)
USE_AMP               = True       # bf16 on A100

# Trajectory cache dtype on CPU (keeps RAM modest)
EMBEDDER_CACHE_DTYPE  = torch.float16

# Per-tx token tensors for the temporal head (shape [N_tx, W, T] where W = MAX_IN+MAX_OUT)
MAX_IN_WALLETS        = 4          # input addresses kept per tx (most-active first)
MAX_OUT_WALLETS       = 4          # output addresses kept per tx
MAX_TRAJ              = 4          # reduced from 16: fill_rate at T=16 was 4.4%, so most was padding

# Temporal head hyperparameters (single cross-wallet attention pool with TX as query)
TEMPORAL_D_MODEL      = 64         # reduced from 128 (overfitting fix)
TEMPORAL_N_HEADS      = 4
TEMPORAL_DROPOUT      = 0.5        # raised from 0.2 (overfitting fix)
TEMPORAL_LR           = 5e-4
TEMPORAL_WEIGHT_DECAY = 1e-3       # raised from 5e-4 (overfitting fix)
TEMPORAL_EPOCHS       = 30
TEMPORAL_BATCH_SIZE   = 1024
ADV_LAMBDA            = 0.1        # 0 disables; >0 enables domain-adversarial head on h_T^final
                                   # (gradient reversal predicts normalised timestep -> drift mitigation)
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")


## 1. Load data, build the heterogeneous graph, and define causal masks

We build:
- `tx_X_std`, `tx_label`, `tx_time`: per-tx standardised features (scaler fit on train timesteps only), labels, and timesteps.
- `train_idx`, `val_idx`, `test_idx`: temporal split indices over labelled txs.
- `hetero_graph`: static `HeteroData` with `wallet` (822K) and `tx` (~204K) node types, the bipartite addr↔tx relations and their reverses, and the within-timestep tx↔tx relation. **Wallet input features are placeholder zeros** (the lifetime-feature file is excluded by the leak rules above; wallet embeddings emerge purely from message passing).
- `wallet_first_seen[w]`: minimum tx-time over edges incident to wallet `w`. Determines when `w` becomes "known".
- `edge_active_mask[rel][t]`: per-relation, per-snapshot boolean mask over the static `edge_index` selecting only edges where every endpoint with a timestamp has `≤ t`.
- `incident_in_per_tx`, `incident_out_per_tx`: per-tx lists of incident input/output wallets.
- `wallet_active_t`: per-wallet sorted list of timesteps the wallet was incident to some tx (used to build trajectories).


In [ ]:
def map_edges(edge_df, src_col, dst_col, src_map, dst_map):
    src = edge_df[src_col].map(src_map); dst = edge_df[dst_col].map(dst_map)
    valid = src.notna() & dst.notna()
    return torch.tensor(np.stack([
        src[valid].astype(np.int64).values,
        dst[valid].astype(np.int64).values,
    ]), dtype=torch.long)

# === Tx features and labels ===
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_tx = len(tx_feat_cols)

merged = tx_features_df[["txId", "Time step"]].merge(tx_classes_df[["txId", "label"]], on="txId", how="left")
merged["label"] = merged["label"].fillna(-1).astype(np.int8)
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values

# === Temporal split (train: t<=TRAIN_END, val: (TRAIN_END, VAL_END], test: > VAL_END) ===
labelled = (tx_label != -1)
train_idx = np.where(labelled & (tx_time <= TRAIN_END))[0]
val_idx   = np.where(labelled & (tx_time > TRAIN_END) & (tx_time <= VAL_END))[0]
test_idx  = np.where(labelled & (tx_time > VAL_END))[0]
print(f"labelled txs: {labelled.sum():,}  illicit_rate(all-labelled)={tx_label[labelled].mean():.4f}")
print(f"  train (t<= {TRAIN_END}):           {len(train_idx):,}   illicit_rate={tx_label[train_idx].mean():.4f}")
print(f"  val   ({TRAIN_END} < t <= {VAL_END}):  {len(val_idx):,}     illicit_rate={tx_label[val_idx].mean():.4f}")
print(f"  test  (t >  {VAL_END}):           {len(test_idx):,}   illicit_rate={tx_label[test_idx].mean():.4f}")

# === Standardise tx features (fit on TRAIN ONLY) ===
tx_scaler = StandardScaler().fit(tx_X_raw[train_idx])
tx_X_std  = tx_scaler.transform(tx_X_raw).astype(np.float32)

# === Wallet space (no static features used; just the address->idx map) ===
wallet_classes_df = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_classes_df["address"].values)}
N_w = len(wallet_classes_df)

# === Edges ===
addr_tx_edges_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")    # input_address -> tx
tx_addr_edges_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")    # tx -> output_address
tx_tx_edges_df   = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")  # tx -> tx

# === Per-edge timestep helpers (for masks and for first_seen/active_t computation) ===
def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]  = df[addr_col].map(wallet_addr_to_idx)
    df["tx"] = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]  = df["w"].astype(np.int64)
    df["tx"] = df["tx"].astype(np.int64)
    df["t"]  = tx_time[df["tx"].values]
    return df[["w", "tx", "t"]]

at_df = edges_with_time(addr_tx_edges_df, "input_address",  "txId")  # input wallet -> tx
ta_df = edges_with_time(tx_addr_edges_df, "output_address", "txId")  # tx -> output wallet

# === Per-wallet first_seen (min tx-time over incident addr↔tx edges) ===
wallet_first_seen = np.full(N_w, N_TIME_STEPS + 1, dtype=np.int64)
for sub in (at_df, ta_df):
    grp = sub.groupby("w")["t"].min()
    cur = wallet_first_seen[grp.index.values.astype(np.int64)]
    wallet_first_seen[grp.index.values.astype(np.int64)] = np.minimum(cur, grp.values.astype(np.int64))

# === Per-wallet active timesteps (sorted ascending), used to build trajectories ===
wallet_active_t = defaultdict(list)
for sub in (at_df, ta_df):
    for w, t in zip(sub["w"].values, sub["t"].values):
        wallet_active_t[int(w)].append(int(t))
for w in list(wallet_active_t.keys()):
    wallet_active_t[w] = sorted(set(wallet_active_t[w]))
N_wallet_events = sum(len(v) for v in wallet_active_t.values())
print(f"wallet appearance events (unique (w,t) pairs across addr<->tx edges): {N_wallet_events:,}")
print(f"  unique wallets touched: {len(wallet_active_t):,}")

# === Per-tx incident wallets (input / output) ===
incident_in_per_tx  = defaultdict(list)
incident_out_per_tx = defaultdict(list)
for tx, w in zip(at_df["tx"].values, at_df["w"].values):
    incident_in_per_tx[int(tx)].append(int(w))
for tx, w in zip(ta_df["tx"].values, ta_df["w"].values):
    incident_out_per_tx[int(tx)].append(int(w))

# === Hetero graph (static topology; causality enforced via per-snapshot edge masks) ===
hetero_graph = HeteroData()
# Wallet input is a placeholder zero feature (single channel) — no leaky lifetime features.
hetero_graph["wallet"].x = torch.zeros(N_w, 1, dtype=torch.float32)
hetero_graph["tx"].x     = torch.from_numpy(tx_X_std)

hetero_graph["wallet", "addr_to_tx", "tx"].edge_index = map_edges(
    addr_tx_edges_df, "input_address", "txId", wallet_addr_to_idx, tx_id_to_idx)
hetero_graph["tx", "tx_to_addr", "wallet"].edge_index = map_edges(
    tx_addr_edges_df, "txId", "output_address", tx_id_to_idx, wallet_addr_to_idx)
hetero_graph["tx", "tx_to_tx", "tx"].edge_index = map_edges(
    tx_tx_edges_df, "txId1", "txId2", tx_id_to_idx, tx_id_to_idx)
hetero_graph = ToUndirected()(hetero_graph)

# === Per-relation per-snapshot edge active masks ===
# An edge (s, d) with src-type S, dst-type D is active at snapshot t iff
#   t_node[S][s] <= t  AND  t_node[D][d] <= t
# where t_node["tx"]     = tx_time
#       t_node["wallet"] = wallet_first_seen   (the wallet has been observed in some tx by t)
def t_of_node(ntype):
    if ntype == "tx":     return torch.from_numpy(tx_time)
    if ntype == "wallet": return torch.from_numpy(wallet_first_seen)
    raise ValueError(ntype)

print("\nHetero graph (actor + tx-tx, AddrAddr excluded for safety):")
edge_active_mask = {}
for etype in hetero_graph.edge_types:
    s_t, _, d_t = etype
    src, dst = hetero_graph[etype].edge_index[0], hetero_graph[etype].edge_index[1]
    src_t = t_of_node(s_t)[src]
    dst_t = t_of_node(d_t)[dst]
    mask = torch.zeros(N_TIME_STEPS + 1, src.size(0), dtype=torch.bool)
    for t in range(1, N_TIME_STEPS + 1):
        mask[t] = (src_t <= t) & (dst_t <= t)
    edge_active_mask[etype] = mask
    print(f"  {str(etype):60s} edges={src.size(0):>9,}")
for ntype in hetero_graph.node_types:
    print(f"  {ntype:8s} nodes={hetero_graph[ntype].num_nodes:>9,}  features={hetero_graph[ntype].x.shape[1]}")

# === Move static graph + masks to device; keep raw arrays available on CPU too ===
hetero_graph     = hetero_graph.to(DEVICE)
edge_active_mask = {k: v.to(DEVICE) for k, v in edge_active_mask.items()}
tx_X_std_dev     = torch.from_numpy(tx_X_std).to(DEVICE)
tx_time_dev      = torch.from_numpy(tx_time).to(DEVICE)
tx_label_dev     = torch.from_numpy(tx_label).to(DEVICE)

F_wallet_in = 1
print(f"\nF_tx={F_tx}  F_wallet_in={F_wallet_in}  N_tx={N_tx:,}  N_w={N_w:,}")


## 2. Tabular baselines: Random Forest and Logistic Regression

Per-tx 165 features only, temporal split as defined above. Class-balanced weights. These should land near the original Elliptic paper's RF (~0.83 illicit F1 in the easy regime / ~0.5 once the t = 43 distribution shift hits the test set).


In [ ]:
results = {}

def report(y_true, y_pred, y_proba, name):
    f1  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_true, y_proba) if y_proba is not None else float("nan")
    print(f"[{name}] illicit F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=["licit", "illicit"], digits=4, zero_division=0))
    results[name] = {"f1": f1, "auc": auc}
    return f1, auc

X_train = tx_X_std[train_idx]; y_train = tx_label[train_idx]
X_val   = tx_X_std[val_idx];   y_val   = tx_label[val_idx]
X_test  = tx_X_std[test_idx];  y_test  = tx_label[test_idx]

print("=== Random Forest ===")
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
report(y_test, rf.predict(X_test), rf.predict_proba(X_test)[:, 1], "RF")

print("\n=== Logistic Regression ===")
lr = LogisticRegression(class_weight="balanced", max_iter=2000, solver="saga", n_jobs=-1, random_state=RANDOM_SEED)
lr.fit(X_train, y_train)
report(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:, 1], "LR")


## 3. Heterogeneous GNN embedder, causally trained

Architecture: `EMBEDDER_NUM_LAYERS` stacked `HeteroConv` layers, conv type chosen by `EMBEDDER_CONV_TYPE`:
- `"sage"` — `SAGEConv((-1, -1), hidden_dim)` on every relation. Handles same-type and bipartite identically.
- `"gcn"` — `GCNConv(hidden_dim, hidden_dim, add_self_loops=False)` on **same-type** relations (here only `tx ↔ tx`), `SAGEConv` on the bipartite addr↔tx relations (vanilla GCN is undefined on bipartite). Mirrors the build_conv pattern in `wallet_sota.ipynb`.

Each `HeteroConv` is followed by per-node-type `LayerNorm`, ReLU, dropout. A linear head on `tx` hidden states produces a 2-class logit per tx.

**Per-snapshot causal forward**:
1. Restrict every relation's `edge_index` to `edge_active_mask[rel][t]` — only edges where every endpoint with a timestamp has timestep `≤ t`.
2. Zero out tx input features for txs with `t' > t` so they cannot propagate forward through any 2-hop wallet path.
3. Wallet input is a constant zero placeholder (no lifetime features touched).

**Per-snapshot supervision**:
For each `t` in `[1, TRAIN_END]`, compute loss only on tx labels at exactly `t` (and only for txs in the train index). The embedder never sees a label with `t' > t` in its loss term.

**Caching**:
After training, run inference snapshots `t = 1..N_TIME_STEPS` and persist:
- `tx_emb_cache[i]` for tx `i` at snapshot `t = tx_time[i]` (one embedding per tx, anchored to its own timestep).
- `wallet_emb_cache[event_row]` for each `(wallet, t)` event, where `event_row = event_to_row[(w, t)]`.

Caches are stored as fp16 on CPU to keep RAM manageable; per-batch lookups copy the relevant slice to GPU during transformer training.


In [ ]:
class HeteroEmbedder(nn.Module):
    """Hetero GNN over (wallet, tx) graph with switchable conv_type.
    Outputs per-node hidden states + per-tx logits."""
    def __init__(self, edge_types, wallet_in_dim, tx_in_dim,
                 conv_type="sage", hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        if conv_type not in ("gcn", "sage"):
            raise ValueError(f"conv_type must be 'gcn' or 'sage', got {conv_type!r}")
        self.conv_type = conv_type
        self.input_proj = nn.ModuleDict({
            "wallet": nn.Linear(wallet_in_dim, hidden_dim),
            "tx":     nn.Linear(tx_in_dim,     hidden_dim),
        })
        def build_conv(rel):
            src_t, _, dst_t = rel
            # GCN is undefined on bipartite; fall back to SAGE for cross-type relations.
            if src_t == dst_t and conv_type == "gcn":
                return GCNConv(hidden_dim, hidden_dim, add_self_loops=False)
            return SAGEConv((-1, -1), hidden_dim)
        self.layers = nn.ModuleList([
            HeteroConv({rel: build_conv(rel) for rel in edge_types}, aggr="sum")
            for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.ModuleDict({nt: nn.LayerNorm(hidden_dim) for nt in ("wallet", "tx")})
            for _ in range(num_layers)
        ])
        self.tx_classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict, return_embeddings=False):
        h = {nt: self.input_proj[nt](x) for nt, x in x_dict.items()}
        for layer, norm in zip(self.layers, self.norms):
            h = layer(h, edge_index_dict)
            h = {nt: F.relu(norm[nt](v)) for nt, v in h.items()}
            h = {nt: F.dropout(v, p=self.dropout, training=self.training) for nt, v in h.items()}
        tx_logits = self.tx_classifier(h["tx"])
        if return_embeddings:
            return tx_logits, h
        return tx_logits


In [ ]:
print(f"\n=== Training causal hetero embedder "
      f"({EMBEDDER_NUM_LAYERS} {EMBEDDER_CONV_TYPE.upper()} layers, hidden={EMBEDDER_HIDDEN_DIM}) ===")
embedder = HeteroEmbedder(
    edge_types=hetero_graph.edge_types,
    wallet_in_dim=F_wallet_in, tx_in_dim=F_tx,
    conv_type=EMBEDDER_CONV_TYPE,
    hidden_dim=EMBEDDER_HIDDEN_DIM, num_layers=EMBEDDER_NUM_LAYERS, dropout=EMBEDDER_DROPOUT,
).to(DEVICE)
optim_e = torch.optim.Adam(embedder.parameters(), lr=EMBEDDER_LR, weight_decay=EMBEDDER_WEIGHT_DECAY)

# Per-tx in-set masks on device
train_set_dev = torch.zeros(N_tx, dtype=torch.bool, device=DEVICE)
train_set_dev[torch.from_numpy(train_idx).to(DEVICE)] = True
val_set_dev   = torch.zeros(N_tx, dtype=torch.bool, device=DEVICE)
val_set_dev[torch.from_numpy(val_idx).to(DEVICE)]   = True

# Class weight from training rows (illicit upweighted)
n_pos_tr = int((tx_label[train_idx] == 1).sum())
n_neg_tr = int((tx_label[train_idx] == 0).sum())
class_weight = torch.tensor([1.0, n_neg_tr / max(n_pos_tr, 1)], dtype=torch.float, device=DEVICE)
print(f"train: n_pos={n_pos_tr:,}  n_neg={n_neg_tr:,}  illicit_weight={class_weight[1]:.2f}")

amp_enabled = USE_AMP and DEVICE.type == "cuda"
amp_dtype   = torch.bfloat16 if amp_enabled else torch.float32

def run_snapshot(t, train_mode):
    """One forward pass for the causal snapshot at timestep t."""
    edge_index_t = {rel: hetero_graph[rel].edge_index[:, edge_active_mask[rel][t]]
                    for rel in hetero_graph.edge_types}
    # Zero out features for txs at t' > t so they can't propagate via 2-hop paths
    active_tx_mask = (tx_time_dev <= t).float().unsqueeze(-1)
    tx_x_t = tx_X_std_dev * active_tx_mask
    x_dict = {"wallet": hetero_graph["wallet"].x, "tx": tx_x_t}
    if train_mode: embedder.train()
    else:          embedder.eval()
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx, torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=amp_enabled):
        logits, h = embedder(x_dict, edge_index_t, return_embeddings=True)
    return logits, h

# Per-snapshot SGD: one backward per snapshot (txs at exactly t in train set contribute to loss)
for epoch in range(1, EMBEDDER_EPOCHS + 1):
    t0 = time.time()
    ep_loss, ep_n = 0.0, 0
    for t in range(1, TRAIN_END + 1):
        m = (tx_time_dev == t) & train_set_dev
        if not m.any():
            continue
        optim_e.zero_grad()
        logits, _ = run_snapshot(t, train_mode=True)
        loss = F.cross_entropy(logits[m].float(), tx_label_dev[m], weight=class_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(embedder.parameters(), 1.0)
        optim_e.step()
        ep_loss += float(loss.detach()) * int(m.sum())
        ep_n    += int(m.sum())
    # Quick val (run snapshot at each val timestep, classify val txs at exactly that t)
    val_y, val_p = [], []
    with torch.no_grad():
        for t in range(TRAIN_END + 1, VAL_END + 1):
            logits, _ = run_snapshot(t, train_mode=False)
            m = (tx_time_dev == t) & val_set_dev
            if not m.any():
                continue
            prob = logits[m].float().softmax(-1)[:, 1].cpu().numpy()
            val_y.append(tx_label_dev[m].cpu().numpy())
            val_p.append(prob)
    if val_y:
        vy = np.concatenate(val_y); vp = np.concatenate(val_p)
        v_f1 = f1_score(vy, (vp >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
    else:
        v_f1 = 0.0
    print(f"epoch {epoch:2d}  train loss={ep_loss/max(ep_n,1):.4f}  val F1@0.5={v_f1:.4f}  ({time.time()-t0:.1f}s)")

print("\n=== Building causal embedding cache ===")
# Index events: row in cache <-> (wallet_idx, t)
event_to_row = {}
event_t_arr  = np.zeros(N_wallet_events, dtype=np.int64)
event_w_arr  = np.zeros(N_wallet_events, dtype=np.int64)
ev = 0
for w, ts in wallet_active_t.items():
    for t in ts:
        event_w_arr[ev] = w
        event_t_arr[ev] = t
        event_to_row[(w, t)] = ev
        ev += 1
# Per-timestep grouping for fast cache fills
events_at_t = [np.empty(0, dtype=np.int64) for _ in range(N_TIME_STEPS + 1)]
for tt in range(1, N_TIME_STEPS + 1):
    events_at_t[tt] = np.where(event_t_arr == tt)[0]

tx_emb_cache_cpu     = torch.zeros(N_tx, EMBEDDER_HIDDEN_DIM, dtype=EMBEDDER_CACHE_DTYPE)
wallet_emb_cache_cpu = torch.zeros(N_wallet_events, EMBEDDER_HIDDEN_DIM, dtype=EMBEDDER_CACHE_DTYPE)

embedder.eval()
with torch.no_grad():
    for t in range(1, N_TIME_STEPS + 1):
        _, h = run_snapshot(t, train_mode=False)
        # Save tx embeddings for txs at exactly t (their causal "as-of" embedding is computed here)
        m_tx = (tx_time_dev == t)
        tx_idx_t = torch.nonzero(m_tx, as_tuple=False).squeeze(-1)
        if tx_idx_t.numel() > 0:
            tx_emb_cache_cpu[tx_idx_t.cpu()] = h["tx"][tx_idx_t].to(EMBEDDER_CACHE_DTYPE).cpu()
        # Save wallet embeddings for wallets active at exactly t
        evs = events_at_t[t]
        if evs.size > 0:
            ws = torch.from_numpy(event_w_arr[evs]).to(DEVICE)
            wallet_emb_cache_cpu[evs] = h["wallet"][ws].to(EMBEDDER_CACHE_DTYPE).cpu()
        if t % 5 == 0 or t == N_TIME_STEPS:
            print(f"  cached snapshot t={t}")

print(f"tx cache: {tuple(tx_emb_cache_cpu.shape)}  wallet event cache: {tuple(wallet_emb_cache_cpu.shape)}")


## 4. Single cross-wallet attention head with $T$ as query (+ optional drift adversary)

Earlier iterations had a per-wallet attention pool stacked under a cross-wallet pool. With `MAX_TRAJ=16` the trajectory grid was ~4.4% non-pad — the per-wallet pool was attending over 1-element sequences for the majority of slots and was effectively identity. We've simplified to a **single** cross-wallet attention pool over the flattened `W × T` token grid.

Per-tx token tensor: `[N_tx, W, T]` with `W = MAX_IN + MAX_OUT` wallet slots and `T = MAX_TRAJ` trajectory positions per slot. Slot `i ∈ [0, MAX_IN)` is an input wallet; slot `i ∈ [MAX_IN, W)` is an output wallet. Each slot stores up to `MAX_TRAJ` most-recent observations with `t' ≤ target_t`.

**Time encoding** (TGAT / Bochner): `TimeEncoding` with learnable frequencies $\omega_1, \ldots, \omega_{d/2}$ produces $\phi(\tau) = [\cos(\omega_1 \tau), \sin(\omega_1 \tau), \ldots]$. Continuous in $\tau$, so test timesteps `t ∈ [35, 49]` are well-defined even though the embedder loop only gradient-updates `t ≤ TRAIN_END`.

**Architecture**:
1. Build flat key/value sequence `[B, W*T, d]`: each token = `wallet_proj(h_w^{(\tau_j)}) + φ(τ_j) + seg_emb`. Pad mask = `slot_pad ∨ traj_pad`.
2. Single MultiheadAttention with query = `tx_emb_proj(h_T^{(t)}) + φ(t)`, attending over the flat sequence → context `c ∈ ℝ^d`.
3. Residual: `h_T_final = LN(tx_q + c)`. Classifier head sees `[h_T_final ‖ tx_feat_proj(tx_features)]` → 2-layer MLP → 2 logits.

**Optional domain-adversarial head**: `timestep_adv` predicts normalised `target_t / N_TIME_STEPS` from `h_T_final` through a gradient-reversal layer. Combined loss = `cls_loss + adv_loss` (gradient reversal makes the embedder trunk *fight* the adversary, pushing `h_T_final` toward timestep-invariant features — the standard DANN-style drift-mitigation trick). Toggle via `ADV_LAMBDA`; set `0.0` to disable.

Token tensor shapes (per tx) — unchanged:
```
token_traj_row  [W, T] int64    row in wallet_emb_cache_cpu; -1 = pad
token_traj_t    [W, T] int16    observation τ; 0 = pad
token_traj_pad  [W, T] bool     True = pad (no such trajectory entry for this slot)
token_seg       [W]    int8     0 = INPUT wallet, 1 = OUTPUT wallet
token_wallet_pad[W]    bool     True = pad (this slot has no incident wallet)
```


In [ ]:
# === Per-tx token tensors with shape [N_tx, W, T] ===
W = MAX_IN_WALLETS + MAX_OUT_WALLETS
T = MAX_TRAJ
print(f"per-tx token grid: W={W}  T={T}")

token_traj_row  = np.full((N_tx, W, T), -1, dtype=np.int64)   # row in wallet_emb_cache_cpu; -1 = pad
token_traj_t    = np.zeros((N_tx, W, T), dtype=np.int16)      # observation τ; 0 = pad
token_traj_pad  = np.ones((N_tx, W, T), dtype=bool)           # True = pad
token_seg       = np.zeros((N_tx, W),    dtype=np.int8)       # 0 = INPUT, 1 = OUTPUT
token_wallet_pad= np.ones((N_tx, W),     dtype=bool)          # True = no wallet at this slot

# Wallet "richness" used to pick the top-K most-active wallets per side
wallet_event_count = np.zeros(N_w, dtype=np.int64)
for w, ts in wallet_active_t.items():
    wallet_event_count[w] = len(ts)

def pick_top(wlist, k):
    if not wlist: return []
    if len(wlist) <= k: return list(wlist)
    cnt = wallet_event_count[wlist]
    order = np.argsort(-cnt, kind="stable")
    return [wlist[i] for i in order[:k]]

t0 = time.time()
for tx_idx in range(N_tx):
    target_t = int(tx_time[tx_idx])
    in_w  = pick_top(incident_in_per_tx.get(tx_idx, []),  MAX_IN_WALLETS)
    out_w = pick_top(incident_out_per_tx.get(tx_idx, []), MAX_OUT_WALLETS)
    combined = [(w, 0) for w in in_w] + [(w, 1) for w in out_w]   # (wallet, segment)
    for slot, (w, seg) in enumerate(combined):
        token_seg[tx_idx, slot]        = seg
        token_wallet_pad[tx_idx, slot] = False
        ts = wallet_active_t.get(w, [])
        cut = bisect.bisect_right(ts, target_t)            # entries strictly with τ ≤ target_t kept
        recent = ts[max(0, cut - MAX_TRAJ):cut]            # last-MAX_TRAJ; oldest at j=0, newest at j=len-1
        for j, t_obs in enumerate(recent):
            token_traj_row[tx_idx, slot, j] = event_to_row[(w, t_obs)]
            token_traj_t[tx_idx, slot, j]   = t_obs
            token_traj_pad[tx_idx, slot, j] = False

n_filled_slots = (~token_wallet_pad).sum()
n_filled_traj  = (~token_traj_pad).sum()
print(f"tokens built: traj_row {tuple(token_traj_row.shape)}  "
      f"slot fill_rate={n_filled_slots/(N_tx*W):.3f}  traj fill_rate={n_filled_traj/(N_tx*W*T):.3f}  "
      f"({time.time()-t0:.1f}s)")

# === Sanity check (unit-test for leakage in token construction) ===
# For a sample of txs, verify every populated trajectory entry has τ ≤ target_t.
np.random.seed(0)
sample_idx = np.random.choice(N_tx, size=min(256, N_tx), replace=False)
for tx_idx in sample_idx:
    target_t = int(tx_time[tx_idx])
    valid = ~token_traj_pad[tx_idx]                        # [W, T] bool
    if valid.any():
        max_tau = int(token_traj_t[tx_idx][valid].max())
        assert max_tau <= target_t, \
            f"LEAKAGE: tx_idx={tx_idx} target_t={target_t} but token τ_max={max_tau}"
print("  ✓ leakage sanity check passed: every populated trajectory entry has τ ≤ target_t")

# Pretty-print one example
ex = sample_idx[0]
print(f"\nExample tx_idx={ex}  target_t={tx_time[ex]}  label={tx_label[ex]}")
for slot in range(W):
    if not token_wallet_pad[ex, slot]:
        ts = token_traj_t[ex, slot][~token_traj_pad[ex, slot]].tolist()
        seg_name = "IN " if token_seg[ex, slot] == 0 else "OUT"
        print(f"  slot {slot:2d} [{seg_name}]  τs={ts}")


In [ ]:
class TimeEncoding(nn.Module):
    """TGAT / Bochner sinusoidal time encoding with learnable frequencies.
    Continuous in τ, so timesteps that received no gradient updates during training
    (here: t > TRAIN_END for the embedder loop) still produce well-defined embeddings.
    """
    def __init__(self, d_model: int):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"
        # Init frequencies small so cos/sin stay well-behaved over τ ∈ [1, 49]
        self.freq = nn.Parameter(torch.randn(d_model // 2) * 0.1)

    def forward(self, t):
        # t: any shape (typically [B] or [B, W, T]); float or int.
        x = t.float().unsqueeze(-1) * self.freq                  # [..., d/2]
        return torch.cat([torch.cos(x), torch.sin(x)], dim=-1)   # [..., d]


class GradReverse(torch.autograd.Function):
    """Identity in forward; flips gradient sign (scaled by lambd) in backward.
    Used by the domain-adversarial head: the trunk fights the adversary."""
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class TemporalTxTransformer(nn.Module):
    """Single cross-wallet attention pool with the target tx as query.
    Plus an optional gradient-reversal head that predicts normalised timestep
    from h_T_final, encouraging timestep-invariant final features (drift fix)."""
    def __init__(self, tx_feat_dim, emb_dim, d_model=64, n_heads=4, dropout=0.5):
        super().__init__()
        self.d_model = d_model
        self.tx_emb_proj   = nn.Linear(emb_dim,     d_model)   # h_T^(t)
        self.tx_feat_proj  = nn.Linear(tx_feat_dim, d_model)   # raw per-tx features
        self.wallet_proj   = nn.Linear(emb_dim,     d_model)   # h_w^(τ)
        self.seg_emb       = nn.Embedding(2, d_model)          # 0=INPUT, 1=OUTPUT
        self.time_enc      = TimeEncoding(d_model)

        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 2),
        )
        # Domain-adversarial head: predicts normalised timestep from h_T_final
        self.timestep_adv = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    @staticmethod
    def _safe_attn(attn_module, query, kv, kv_pad):
        """Wrap MHA so all-padded rows return zero instead of NaN-from-softmax."""
        all_pad = kv_pad.all(dim=-1)                              # [B]
        safe_pad = kv_pad.clone()
        safe_pad[all_pad, 0] = False                              # fake-unpad to avoid NaN
        out, _ = attn_module(query, kv, kv, key_padding_mask=safe_pad)
        out = out.squeeze(1)                                      # [B, d]
        out = out.masked_fill(all_pad.unsqueeze(-1), 0.0)
        return out

    def forward(self, tx_feats, tx_emb, target_t,
                wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
                wallet_seg, wallet_pad,
                adv_lambda: float = 0.0, return_adv: bool = False):
        """
        tx_feats        : [B, F_tx]
        tx_emb          : [B, emb_dim]                 (h_T^(t) from embedder cache)
        target_t        : [B]                          (= t_T)
        wallet_traj_emb : [B, W, T, emb_dim]           (cached h_w^(τ); zeros for pad)
        wallet_traj_t   : [B, W, T]                    (observation τ; 0 for pad)
        wallet_traj_pad : [B, W, T]   bool             (True = pad trajectory entry)
        wallet_seg      : [B, W]      long {0, 1}
        wallet_pad      : [B, W]      bool             (True = pad wallet slot)
        adv_lambda      : gradient-reversal scale (only used when return_adv=True)
        return_adv      : if True, also return the adversarial timestep prediction
        """
        B, W, Td, _ = wallet_traj_emb.shape
        d = self.d_model

        # ---- T's representation ----
        tx_q = self.tx_emb_proj(tx_emb) + self.time_enc(target_t)        # [B, d]
        tx_q = self.dropout(tx_q)

        # ---- Build flat key/value sequence [B, W*T, d] ----
        kv = self.wallet_proj(wallet_traj_emb)                            # [B, W, T, d]
        kv = kv + self.time_enc(wallet_traj_t)
        kv = kv + self.seg_emb(wallet_seg).unsqueeze(2)                   # broadcast over T
        kv = self.dropout(kv)
        kv_flat = kv.view(B, W * Td, d)
        # Token is pad if its slot is pad OR its trajectory entry is pad
        slot_pad = wallet_pad.unsqueeze(-1).expand(-1, -1, Td)            # [B, W, T]
        kv_pad   = (slot_pad | wallet_traj_pad).view(B, W * Td)           # [B, W*T]

        # ---- Single cross-wallet attention pool ----
        ctx = self._safe_attn(self.attn, tx_q.unsqueeze(1), kv_flat, kv_pad)   # [B, d]

        # ---- Residual + classifier ----
        h_T_final = self.norm(tx_q + ctx)                                  # [B, d]
        head_in   = torch.cat([h_T_final, self.tx_feat_proj(tx_feats)], dim=-1)
        logits    = self.classifier(head_in)

        if return_adv:
            adv_in  = GradReverse.apply(h_T_final, adv_lambda)
            adv_pred = self.timestep_adv(adv_in).squeeze(-1)               # [B]
            return logits, adv_pred
        return logits


In [ ]:
class TxIdxDataset(Dataset):
    def __init__(self, indices): self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self):           return len(self.indices)
    def __getitem__(self, i):    return int(self.indices[i])

def collate(batch):
    idx = np.asarray(batch, dtype=np.int64)
    # Per-tx side
    tx_feats = torch.from_numpy(tx_X_std[idx])                        # [B, F_tx]
    tx_emb   = tx_emb_cache_cpu[torch.from_numpy(idx)]                 # [B, hidden] fp16
    target_t = torch.from_numpy(tx_time[idx])                          # [B]
    y        = torch.from_numpy(tx_label[idx])                         # [B]
    # Wallet trajectory grid
    rows     = token_traj_row[idx]                                     # [B, W, T] int64; -1=pad
    rows_safe = np.where(rows >= 0, rows, 0)
    wallet_traj_emb = wallet_emb_cache_cpu[torch.from_numpy(rows_safe)]   # [B, W, T, hidden] fp16
    pad_traj_b      = torch.from_numpy(token_traj_pad[idx])            # [B, W, T] bool
    wallet_traj_emb = wallet_traj_emb.masked_fill(pad_traj_b.unsqueeze(-1), 0.0)
    wallet_traj_t   = torch.from_numpy(token_traj_t[idx].astype(np.int64))   # [B, W, T]
    wallet_seg      = torch.from_numpy(token_seg[idx].astype(np.int64))      # [B, W]
    wallet_pad      = torch.from_numpy(token_wallet_pad[idx])                # [B, W] bool

    return (tx_feats, tx_emb.float(), target_t,
            wallet_traj_emb.float(), wallet_traj_t, pad_traj_b,
            wallet_seg, wallet_pad, y)

train_loader = DataLoader(TxIdxDataset(train_idx), batch_size=TEMPORAL_BATCH_SIZE,
                          shuffle=True, collate_fn=collate, num_workers=2)
val_loader   = DataLoader(TxIdxDataset(val_idx),   batch_size=TEMPORAL_BATCH_SIZE * 2,
                          shuffle=False, collate_fn=collate, num_workers=2)
test_loader  = DataLoader(TxIdxDataset(test_idx),  batch_size=TEMPORAL_BATCH_SIZE * 2,
                          shuffle=False, collate_fn=collate, num_workers=2)

model = TemporalTxTransformer(
    tx_feat_dim=F_tx, emb_dim=EMBEDDER_HIDDEN_DIM,
    d_model=TEMPORAL_D_MODEL, n_heads=TEMPORAL_N_HEADS,
    dropout=TEMPORAL_DROPOUT,
).to(DEVICE)
optim_t = torch.optim.AdamW(model.parameters(), lr=TEMPORAL_LR, weight_decay=TEMPORAL_WEIGHT_DECAY)
cw = torch.tensor([1.0, n_neg_tr / max(n_pos_tr, 1)], dtype=torch.float, device=DEVICE)

def best_threshold_f1(y, p):
    if not len(y) or len(np.unique(y)) < 2:
        return 0.5, 0.0
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-10)
    best = int(np.nanargmax(f1s[:-1]))
    return float(thr[best]), float(f1s[best])

def _to_dev(*xs):
    return [x.to(DEVICE) if torch.is_tensor(x) else x for x in xs]

def run_eval(loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            (tx_feats, tx_emb, target_t,
             wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
             wallet_seg, wallet_pad, y) = batch
            (tx_feats, tx_emb, target_t,
             wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
             wallet_seg, wallet_pad) = _to_dev(
                 tx_feats, tx_emb, target_t,
                 wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
                 wallet_seg, wallet_pad)
            logits = model(tx_feats, tx_emb, target_t,
                           wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
                           wallet_seg, wallet_pad)
            ys.append(y.numpy()); ps.append(logits.softmax(-1)[:, 1].cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

best_val_f1 = -1.0
best_state  = None
best_epoch  = -1
for epoch in range(1, TEMPORAL_EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_cls, running_adv, seen = 0.0, 0.0, 0
    for batch in train_loader:
        (tx_feats, tx_emb, target_t,
         wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
         wallet_seg, wallet_pad, y) = batch
        (tx_feats, tx_emb, target_t,
         wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
         wallet_seg, wallet_pad, y) = _to_dev(
             tx_feats, tx_emb, target_t,
             wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
             wallet_seg, wallet_pad, y)
        optim_t.zero_grad()
        if ADV_LAMBDA > 0:
            logits, adv_pred = model(tx_feats, tx_emb, target_t,
                                     wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
                                     wallet_seg, wallet_pad,
                                     adv_lambda=ADV_LAMBDA, return_adv=True)
            cls_loss = F.cross_entropy(logits, y, weight=cw)
            adv_target = target_t.float() / float(N_TIME_STEPS)
            adv_loss = F.mse_loss(adv_pred, adv_target)
            loss = cls_loss + adv_loss   # GradReverse flips sign of adv-grad inside the trunk
            running_adv += float(adv_loss.detach()) * y.size(0)
        else:
            logits = model(tx_feats, tx_emb, target_t,
                           wallet_traj_emb, wallet_traj_t, wallet_traj_pad,
                           wallet_seg, wallet_pad)
            cls_loss = F.cross_entropy(logits, y, weight=cw)
            loss = cls_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim_t.step()
        running_cls += float(cls_loss.detach()) * y.size(0)
        seen        += y.size(0)
    vy, vp = run_eval(val_loader)
    v_f1_05 = f1_score(vy, (vp >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
    is_best = v_f1_05 > best_val_f1
    if is_best:
        best_val_f1 = v_f1_05
        best_state  = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch  = epoch
    marker = "  *" if is_best else ""
    extra = f"  adv_loss={running_adv/max(seen,1):.4f}" if ADV_LAMBDA > 0 else ""
    print(f"epoch {epoch:2d}  cls_loss={running_cls/max(seen,1):.4f}{extra}  "
          f"val F1@0.5={v_f1_05:.4f}{marker}  ({time.time()-t0:.1f}s)")

print(f"\nBest epoch (by val F1@0.5): {best_epoch}  ({best_val_f1:.4f})")
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# === Honest reporting at multiple operating points ===
# IMPORTANT: don't calibrate threshold on val — val (t in (29, 34]) has 16.8% illicit
# but test (t > 34) has 6.5% illicit, so a val-tuned threshold is too aggressive on test.
# Instead we report:
#   (a) F1 at fixed threshold 0.5
#   (b) F1 at threshold calibrated on a LATE-TRAIN slice (last 5 train timesteps),
#       which has ~train-distribution illicit rate (closer to test than val is)
#   (c) F1 at the precision-matched operating point (precision >= RF's precision)
#   (d) PR-AUC and ROC-AUC (threshold-free)

# (a) test predictions (saved for diagnostics later)
ty, tp = run_eval(test_loader)
np.save("/tmp/_tx_sota_test_y.npy", ty)
np.save("/tmp/_tx_sota_test_p.npy", tp)
np.save("/tmp/_tx_sota_test_t.npy", tx_time[test_idx])

# (b) late-train threshold
late_train_mask = (tx_time[train_idx] >= TRAIN_END - 4)
late_train_idx  = train_idx[late_train_mask]
late_loader = DataLoader(TxIdxDataset(late_train_idx),
                         batch_size=TEMPORAL_BATCH_SIZE * 2,
                         shuffle=False, collate_fn=collate, num_workers=0)
ly, lp = run_eval(late_loader)
thr_late, _ = best_threshold_f1(ly, lp)
print(f"Late-train threshold (t in [{TRAIN_END-4}, {TRAIN_END}]): {thr_late:.4f}  "
      f"(illicit_rate={ly.mean():.4f})")

# (c) precision-matched operating point (use RF's precision as target)
rf_test_pred  = rf.predict(X_test)
rf_test_proba = rf.predict_proba(X_test)[:, 1]
rf_prec_target = (rf_test_pred[rf_test_pred == 1] == y_test[rf_test_pred == 1]).mean() \
                 if (rf_test_pred == 1).sum() else 0.99
prec_t, rec_t, thr_t = precision_recall_curve(ty, tp)
valid = prec_t[:-1] >= rf_prec_target
if valid.any():
    cand = np.where(valid)[0]
    f1s_c = 2 * prec_t[cand] * rec_t[cand] / (prec_t[cand] + rec_t[cand] + 1e-10)
    best_c = cand[int(np.argmax(f1s_c))]
    thr_pmatch = float(thr_t[best_c])
else:
    thr_pmatch = float(thr_t.max())

from sklearn.metrics import average_precision_score
pr_auc_test = average_precision_score(ty, tp)

print(f"\n=== Test reports for the temporal head ===")
report(ty, (tp >= 0.5).astype(np.int64),       tp, f"Temporal[{EMBEDDER_CONV_TYPE.upper()}] thr=0.5")
report(ty, (tp >= thr_late).astype(np.int64),  tp, f"Temporal[{EMBEDDER_CONV_TYPE.upper()}] thr=late-train")
report(ty, (tp >= thr_pmatch).astype(np.int64),tp, f"Temporal[{EMBEDDER_CONV_TYPE.upper()}] thr=prec-match-RF")
print(f"PR-AUC (threshold-free): {pr_auc_test:.4f}  | RF's test precision target: {rf_prec_target:.4f}")


## 5. Diagnostic ablations

Three small MLP baselines to localise where signal vs. noise enters:
- **MLP[tx_features]** — 165 per-tx features only. If this matches RF, the per-tx tabular signal is the whole story.
- **MLP[tx_emb]** — frozen causal embedder output only. If this is much weaker than tx_features, the embedder is the bottleneck and the temporal head can't fix what's already broken upstream.
- **MLP[tx_features + tx_emb]** — concatenation. If this matches or beats the temporal head, the wallet-trajectory mechanism is **subtractive** and we should drop it.

Each baseline trains a tiny `MLPClassifier(hidden_layer_sizes=(64,))` and reports F1@0.5, F1 at RF's precision, and PR-AUC on the temporal test split.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score

def diag_report(name, y_true, p):
    f1_05 = f1_score(y_true, (p >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
    auc   = roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else float("nan")
    pr_auc = average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else float("nan")
    # F1 at precision >= rf_prec_target
    pr, re, th = precision_recall_curve(y_true, p)
    valid = pr[:-1] >= rf_prec_target
    if valid.any():
        c = np.where(valid)[0]
        f1s = 2 * pr[c] * re[c] / (pr[c] + re[c] + 1e-10)
        f1_pm = float(f1s.max())
    else:
        f1_pm = 0.0
    print(f"[{name:32s}]  F1@0.5={f1_05:.4f}  F1@prec-match={f1_pm:.4f}  AUC={auc:.4f}  PR-AUC={pr_auc:.4f}")
    results[name] = {"f1": f1_05, "auc": auc}

# Build feature matrices
emb_all = tx_emb_cache_cpu.numpy().astype(np.float32)
X_emb_train  = emb_all[train_idx];  X_emb_test  = emb_all[test_idx]
X_both_train = np.concatenate([tx_X_std[train_idx], X_emb_train], axis=1)
X_both_test  = np.concatenate([tx_X_std[test_idx],  X_emb_test],  axis=1)

print("=== Ablation 1: MLP on per-tx features only ===")
mlp_f = MLPClassifier(hidden_layer_sizes=(64,), max_iter=80, random_state=RANDOM_SEED, early_stopping=True)
mlp_f.fit(X_train, y_train)
diag_report("MLP[tx_features]", y_test, mlp_f.predict_proba(X_test)[:, 1])

print("\n=== Ablation 2: MLP on causal-embedder output only (frozen) ===")
mlp_e = MLPClassifier(hidden_layer_sizes=(64,), max_iter=80, random_state=RANDOM_SEED, early_stopping=True)
mlp_e.fit(X_emb_train, y_train)
diag_report("MLP[tx_emb]", y_test, mlp_e.predict_proba(X_emb_test)[:, 1])

print("\n=== Ablation 3: MLP on tx_features + tx_emb ===")
mlp_b = MLPClassifier(hidden_layer_sizes=(64,), max_iter=80, random_state=RANDOM_SEED, early_stopping=True)
mlp_b.fit(X_both_train, y_train)
diag_report("MLP[tx_features+tx_emb]", y_test, mlp_b.predict_proba(X_both_test)[:, 1])


### Per-timestep test F1

Reveals whether the gap between the temporal model and RF lives pre-cliff or post-cliff (the t = 43 distribution shift). If the temporal model is fine pre-cliff (t in [35, 42]) and collapses post-cliff (t in [43, 49]), the story is drift, not capacity.


In [ ]:
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  {'RF F1':>7}  {'Temporal F1':>11}")
print("-" * 44)
test_t_arr = tx_time[test_idx]
for t in range(VAL_END + 1, N_TIME_STEPS + 1):
    m = test_t_arr == t
    if not m.any(): continue
    y_t = y_test[m]
    if y_t.sum() == 0:                     # nothing to score F1 on
        rf_f1, tx_f1 = float("nan"), float("nan")
    else:
        rf_f1 = f1_score(y_t, rf_test_pred[m],            pos_label=1, zero_division=0)
        tx_f1 = f1_score(y_t, (tp[m] >= 0.5).astype(int), pos_label=1, zero_division=0)
    print(f"{t:>3}  {m.sum():>5}  {int(y_t.sum()):>7}  {rf_f1:>7.4f}  {tx_f1:>11.4f}")


## 6. Summary

All numbers below are illicit-class metrics on the **temporal test split** (t > VAL_END). For the temporal head we report F1 at threshold 0.5 (the entry stored in `results`).

Earlier iterations val-calibrated the threshold; this is a measurement trap because val (t in (29, 34]) has 16.8% illicit rate vs. 6.5% on test. We've dropped val-calibration in favour of fixed-0.5, late-train calibration, and precision-matched operating points.


In [ ]:
print(f"{'Model':40s}  {'illicit F1':>10s}  {'AUC':>8s}")
print("-" * 64)
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["f1"]):
    print(f"{name:40s}  {r['f1']:>10.4f}  {r['auc']:>8.4f}")

print("\nNotes:")
print(f"- Temporal split: train t<= {TRAIN_END}, val ({TRAIN_END}, {VAL_END}], test > {VAL_END}.")
print(f"- Embedder: {EMBEDDER_NUM_LAYERS} {EMBEDDER_CONV_TYPE.upper()} layers, hidden={EMBEDDER_HIDDEN_DIM}, "
      f"dropout={EMBEDDER_DROPOUT}, causal edge masks per snapshot.")
print(f"- Single cross-wallet attention pool: d={TEMPORAL_D_MODEL}, heads={TEMPORAL_N_HEADS}, "
      f"W*T={(MAX_IN_WALLETS+MAX_OUT_WALLETS)*MAX_TRAJ} flattened tokens; "
      f"TGAT continuous-time encoding with learnable frequencies.")
print(f"- Drift mitigation: domain-adversarial timestep head with ADV_LAMBDA={ADV_LAMBDA} "
      f"(0 = disabled).")
print("- Reporting: fixed thr=0.5, late-train threshold, precision-matched-to-RF operating point, PR-AUC.")
print("- Wallet lifetime features (`wallets_features.txt`) and AddrAddr edges are excluded to prevent leakage.")
